# Przetwarzanie wstępne. Filtracja kontekstowa.


### Cel:
- zapoznanie z pojęciem kontekstu / filtracji kontekstowej,
- zapoznanie z pojęciem konwolucji (splotu),
- zapoznanie z wybranymi filtrami:
	- filtry liniowe dolnoprzepustowe:
		- filtr uśredniający,
		- filtr Gaussa.
	- filtry nielinowe:
		- mediana,
		- mediana dla obrazów kolorowych.
	- filtry liniowe górnoprzepustowe:
			- laplasjan,
			- operator Robersta, Prewitta, Sobela.
- zadanie domowe: adaptacyjna filtracja medianowa.

### Filtry liniowe uśredniające (dolnoprzepustowe)

Jest to podstawowa rodzina filtrów stosowana w cyfrowym przetwarzaniu obrazów.
Wykorzystuje się je w celu "rozmazania" obrazu i tym samym redukcji szumów (zakłóceń) na obrazie.
Filtr określony jest przez dwa parametry: rozmiar maski (ang. _kernel_) oraz wartości współczynników maski.

Warto zwrócić uwagę, że omawiane w niniejszym rozdziale operacje generują nową wartość piksela na podstawie pewnego fragmentu obrazu (tj. kontekstu), a nie jak operacje punktowe tylko na podstawie jednego piksela.


1. Wczytaj obraz _plansza.png_.
W dalszej części ćwiczenia sprawdzenie działania filtracji dla innych obrazów sprowadzi się do wczytania innego pliku.

2. Podstawowa funkcja to `cv2.filter2D`  - realizacja filtracji konwolucyjnej.
   Proszę sprawdzić jej dokumentację i zwrócić uwagę na obsługę problemu brzegowego (na krawędziach istnieją piksele dla których nie da się wyznaczyć otoczenia).

  Uwaga. Problem ten można też rozwiązać z użyciem funkcji `signal.convolve2d` z biblioteki _scipy_ (`from scipy import signal`).

3. Stwórz podstawowy filtr uśredniający o rozmiarze $3 \times 3$ -- za pomocą funkcji `np.ones`. Wykonaj konwolucję na wczytanym obrazie. Na wspólnym rysunku wyświetl obraz oryginalny, po filtracji oraz moduł z różnicy.

4. Przeanalizuj otrzymane wyniki. Jakie elementy zawiera obraz "moduł z różnicy"? Co na tej podstawie można powiedzieć o filtracji dolnoprzepustowej?

In [ ]:
import cv2
import os
import requests
from matplotlib import pyplot as plt
import numpy as np
from scipy import signal

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/06_Context/'

names = ["jet", "kw", "moon", "lenaSzum", "lena", "plansza"]
images = {}

fileNames = ["jet.png", "kw.png", "moon.png", "lenaSzum.png", "lena.png", "plansza.png"]
for i, fileName in enumerate(fileNames):
    if not os.path.exists(fileName):
        r = requests.get(url + fileName, allow_redirects=True)
        open(fileName, 'wb').write(r.content)
    img = cv2.imread(fileName)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    images[names[i]] = img

In [ ]:
fig, axes = plt.subplots(3, len(images), figsize=(16, 8))

def kernel_filtr(img, size, ax1, ax2, kernel=None):
    if kernel is None:
        kernel = np.ones((size, size), np.float32) / (size * size)
    output = cv2.filter2D(img, -1, kernel)
    ax1.imshow(output, 'gray')
    ax1.axis("off")
    ax1.set_title(f"Po filtrze {size}x{size}")
    modul_z_roznicy = np.abs(img.astype('float32') - output.astype('float32')).astype('uint8')
    ax2.imshow(modul_z_roznicy, 'gray')
    ax2.axis("off")
    ax2.set_title(f"Moduł z różnicy {size}x{size}")

for i, img_name in enumerate(images.keys()):
    img = images[img_name]
    axes[0, i].imshow(img, 'gray')
    axes[0, i].axis("off")
    axes[0, i].set_title("original")
    kernel_filtr(img, 3, axes[1, i], axes[2, i])

5. Na wspólnym rysunku wyświetl wyniki filtracji uśredniającej z oknem o rozmiarze 3, 5, 9, 15 i 35.
Wykorzystaj polecenie `plt.subplot`.
Przeanalizuj wpływ rozmiaru maski na wynik.

In [ ]:
cases_kernel = (3, 5, 9, 15, 35)

fig, axes = plt.subplots(2, len(cases_kernel), figsize=(16, 6))
img = images["plansza"]

for i in range(len(cases_kernel)):
    size = cases_kernel[i]
    kernel_filtr(img, size, axes[0, i], axes[1, i])
    

6. Wczytaj obraz _lena.png_.
Zaobserwuj efekty filtracji dolnoprzepustowej dla obrazu rzeczywistego.

In [ ]:
cases_kernel = (3, 5, 9, 15, 35)

fig, axes = plt.subplots(2, len(cases_kernel), figsize=(16, 6))
img = images["lena"]

for i in range(len(cases_kernel)):
    size = cases_kernel[i]
    kernel_filtr(img, size, axes[0, i], axes[1, i])

7. Niekorzystny efekt towarzyszący wykonanym filtracjom dolnoprzepustowym to utrata ostrości.
Częściowo można go zniwelować poprzez odpowiedni dobór maski.
Wykorzystaj maskę:  `M = np.array([1 2 1; 2 4 2; 1 2 1])`.
Przed obliczeniami należy jeszcze wykonać normalizację - podzielić każdy element maski przez sumę wszystkich elementów: `M = M/sum(sum(M));`.
Tak przygotowaną maskę wykorzystaj w konwolucji - wyświetl wyniki tak jak wcześniej.
Możliwe jest też wykorzystywanie innych masek - współczynniki można dopasowywać do konkretnego problemu.

In [ ]:
M = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]])
M = M/sum(sum(M))

fig, axes = plt.subplots(3, len(images), figsize=(16, 8))

for i, img_name in enumerate(images.keys()):
    img = images[img_name]
    axes[0, i].imshow(img, 'gray')
    axes[0, i].axis("off")
    axes[0, i].set_title("original")
    kernel_filtr(img, 3, axes[1, i], axes[2, i], M)

8. Skuteczną i często wykorzystywaną maską jest tzw. maska Gasussa.
Jest to zbiór liczb, które aproksymują dwuwymiarowy rozkład Gaussa.
Parametrem jest odchylenie standardowe i rozmiar maski.

9. Wykorzystując przygotowaną funkcję `fgaussian` stwórz maskę o rozmiarze $5 \times 5$ i odchyleniu standardowym 0.5.
  Wykorzystując funkcję `mesh` zwizualizuj filtr.
  Sprawdź jak parametr ``odchylenie standardowe'' wpływa na ``kształt'' filtru.

  Uwaga. W OpenCV dostępna jest *dedykowana* funkcja do filtracji Gaussa - `GaussianBlur`.
  Proszę na jednym przykładzie porównać jej działanie z użytym wyżej rozwiązaniem.

10. Wykonaj filtrację dla wybranych (2--3) wartości odchylenia standardowego.


In [ ]:
def fgaussian(size, sigma):
     m = n = size
     h, k = m//2, n//2
     x, y = np.mgrid[-h:h+1, -k:k+1]
     g = np.exp(-(x**2 + y**2)/(2*sigma**2))
     return g /g.sum()


def mesh(kernel, size):
    fig = plt.figure()
    ax = fig.add_subplot(projection = '3d')


    X = np.arange(-size//2, size//2, 1)
    Y = np.arange(-size//2, size//2, 1)
    X, Y = np.meshgrid(X, Y)
    Z = kernel

    ax.plot_surface(X, Y, Z)

    plt.show()


In [ ]:
gaussian_sigma_cases = (0.3, 0.5, 0.7)
size = 5

for sigma_case in gaussian_sigma_cases:
    kernel = fgaussian(size, sigma_case)
    print(f"Odchylenie standardowe: {sigma_case}")
    mesh(kernel, size)
    fig, axes = plt.subplots(3, len(images), figsize=(16, 8))
    print(f"Odchylenie standardowe: {sigma_case}")
    for i, img_name in enumerate(images.keys()):
        img = images[img_name]
        axes[0, i].imshow(img, 'gray')
        axes[0, i].axis("off")
        axes[0, i].set_title("original")
        kernel_filtr(img, size, axes[1, i], axes[2, i], kernel)
    plt.show()
    fig, axes = plt.subplots(1, len(images), figsize=(16, 6))
    for i, img_name in enumerate(images.keys()):
        img = images[img_name]
        img_gb = cv2.GaussianBlur(img, (size, size), sigmaX=sigma_case)
        axes[i].imshow(img_gb, 'gray')
        axes[i].axis("off")
        axes[i].set_title("GaussianBlur")
    plt.show()

### Filtry nieliniowe -- mediana

Filtry rozmywające redukują szum, ale niekorzystnie wpływają na ostrość obrazu.
Dlatego często wykorzystuje się filtry nieliniowe - np. filtr medianowy (dla przypomnienia: mediana - środkowa wartość w posortowanym ciągu liczb).

Podstawowa różnica pomiędzy filtrami liniowymi, a nieliniowymi polega na tym, że przy filtracji liniowej na nową wartość piksela ma wpływ wartość wszystkich pikseli z otoczenia (np. uśrednianie, czasem ważone), natomiast w przypadku filtracji nieliniowej jako nowy piksel wybierana jest któraś z wartości otoczenia - według jakiegoś wskaźnika (wartość największa, najmniejsza czy właśnie mediana).


1. Wczytaj obraz _lenaSzum.png_ (losowe 10% pikseli białych lub czarnych - tzw. zakłócenia impulsowe). Przeprowadź filtrację uśredniającą z rozmiarem maski 3x3. Wyświetl, podobnie jak wcześniej, oryginał, wynik filtracji i moduł z różnicy. Wykorzystując funkcję ``cv2.medianBlur` wykonaj filtrację medianową _lenaSzum.png_ (z rozmiarem maski $3 \times 3$). Wyświetl, podobnie jak wcześniej, oryginał, wynik filtracji i moduł z różnicy. Która filtracja lepiej radzi sobie z tego typu szumem?

  Uwaga. Taki sam efekt da również użycie funkcji `signal.medfilt2d`.


In [ ]:
size = 3

def compare_usredniajaca_medianowa(img):
    fig, axes = plt.subplots(1, 5, figsize=(16, 8))
    axes[0].imshow(img, 'gray')
    axes[0].axis("off")
    axes[0].set_title("original")
    kernel_filtr(img, size, axes[1], axes[2])
    img_gb = cv2.medianBlur(img, size)
    axes[3].imshow(img_gb, 'gray')
    axes[3].axis("off")
    axes[3].set_title("medianBlur")
    img_mod = np.abs(img_gb.astype('float32') -  img.astype('float32')).astype('uint8')
    axes[4].imshow(img_mod, 'gray')
    axes[4].axis("off")
    axes[4].set_title("medianBlur modulo")

    plt.show()

compare_usredniajaca_medianowa(images["lenaSzum"])

2. Przeprowadź filtrację uśredniającą, a następnie medianową obrazu _lena.png_.
   Wyniki porównaj - dla obu wyświetl: oryginał, wynik filtracji i moduł z różnicy.
   Szczególną uwagę zwróć na ostrość i krawędzie.
   W której filtracji krawędzie zostają lepiej zachowane?

In [ ]:
compare_usredniajaca_medianowa(images["lena"])

In [ ]:
def do_medianaBlur_n_times(img, n, kernel_size=3):
    total = n + 1
    cols = 5
    rows = int(np.ceil(total / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 8))
    axes = axes.ravel()
    axes[0].imshow(img, cmap='gray')
    axes[0].axis("off")
    axes[0].set_title("original")
    img_temp = img
    for i in range(1, total):
        img_temp = cv2.medianBlur(img_temp, kernel_size)
        axes[i].imshow(img_temp, cmap='gray')
        axes[i].axis("off")
        axes[i].set_title(f"medianBlur {i}")
    for j in range(total, rows*cols):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

do_medianaBlur_n_times(images["lena"], 10)


3. Ciekawy efekt można uzyskać wykonując filtrację medianową wielokrotnie. Określa się go mianem  posteryzacji.  W wyniku przetwarzania z obrazka usunięte zostają detale, a duże obszary uzyskują tą samą wartość jasności.  Wykonaj operację mediany $5 \times 5$ na obrazie _lena.png_ 10-krotnie. (wykorzystaj np. pętlę `for`).


Inne filtry nieliniowe:
- filtr modowy - moda (dominanta) zamiast mediany,
- filtr olimpijski - średnia z podzbioru otoczenia (bez wartości ekstremalnych),
- hybrydowy filtr medianowy - mediana obliczana osobno w różnych podzbiorach otoczenia (np. kształt ``x'',``+''), a jako wynik brana jest mediana ze zbioru wartość elementu centralnego, mediana z ``x'' i mediana z ``+'',
- filtr minimalny i maksymalny (będą omówione przy okazji operacji morfologicznych w dalszej części kursu).


Warto zdawać sobie sprawę, z szerokich możliwości dopasowywania rodzaju filtracji do konkretnego rozważanego problemu i rodzaju zaszumienia występującego na obrazie.

## Filtry liniowe górnoprzepustowe (wyostrzające, wykrywające krawędzie)

Zadaniem filtrów górnoprzepustowych jest wydobywanie z obrazu składników odpowiedzialnych za szybkie zmiany jasności - konturów, krawędzi, drobnych elementów tekstury.

### Laplasjan (wykorzystanie drugiej pochodnej obrazu)

1. Wczytaj obraz _moon.png_.

2. Wprowadź podstawową maskę laplasjanu:
\begin{equation}
M =
\begin{bmatrix}
0 & 1& 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0
\end{bmatrix}
\end{equation}

3. Przed rozpoczęciem obliczeń należy dokonać normalizacji maski - dla rozmiaru $3 \times 3$ podzielić każdy element przez 9.
   Proszę zwrócić uwagę, że nie można tu zastosować takiej samej normalizacji, jak dla filtrów dolnoprzepustowanych, gdyż skutkowałby to dzieleniem przez 0.

4. Wykonaj konwolucję obrazu z maską (`c2.filter2D`). Przed wyświetleniem, wynikowy obraz należy poddać normalizacji (występują ujemne wartości). Najczęściej wykonuje się jedną z dwóch operacji:
- skalowanie (np. poprzez dodatnie 128 do każdego z pikseli),
- moduł (wartość bezwzględna).

Wykonaj obie normalizacje.
Na wspólnym wykresie wyświetl obraz oryginalny oraz przefiltrowany po obu normalizacjach.

In [ ]:
M = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
M = M/(M.shape[0] * M.shape[1])

def compare_filters(img, kernel, name = None):
    moon_conv = cv2.filter2D(img, cv2.CV_64F, kernel)
    moon_conv_scaled = moon_conv + 128
    moon_conv_modulo = np.abs(moon_conv)

    fig, axes = plt.subplots(1, 4, figsize=(16, 8))
    axes[0].axis("off")
    axes[0].set_title("original")
    axes[0].imshow(img, "gray")

    axes[1].axis("off")
    axes[1].set_title("filter")
    if name is not None:
        axes[1].set_title(f"filter: {name}")
    axes[1].imshow(moon_conv, "gray")

    axes[2].axis("off")
    axes[2].set_title("Skalowanie")
    axes[2].imshow(moon_conv_scaled, "gray")

    axes[3].axis("off")
    axes[3].set_title("Modulo")
    axes[3].imshow(moon_conv_modulo, "gray")

    plt.show()

compare_filters(images["moon"], M)

7. Efekt wyostrzenia uzyskuje się po odjęciu/dodaniu (zależy do maski) rezultatu filtracji laplasjanowej i oryginalnego obrazu. Wyświetl na jednym wykresie: obraz oryginalny, sumę oryginału i wyniku filtracji oraz różnicę (bezwzględną) oryginału i wyniku filtracji.
 Uwaga. Aby uniknąć artefaktów, należy obraz wejściowy przekonwertować do formatu ze znakiem.



In [ ]:
moon = images["moon"].astype("int16")
M = np.array([[0, 1, 0],
              [1, -4, 1],
              [0,  1, 0]], dtype=np.int16)


def compare_laplasjan(img, kernel, name = None):
    lap = cv2.filter2D(img, cv2.CV_64F, kernel)
    sharp = img - lap # odejmujemy bo środek maski jest ujmeny 
    diff = np.abs(img - lap)

    sharp_u8 = np.clip(sharp, 0, 255).astype(np.uint8)
    diff_u8  = np.clip(diff, 0, 255).astype(np.uint8)
    lap_u8   = np.clip(lap   , 0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 4, figsize=(16, 8))
    axes[0].axis("off")
    axes[0].set_title("original")
    axes[0].imshow(img, "gray")

    axes[1].axis("off")
    axes[1].set_title("Filtracja")
    if name is not None:
        axes[1].set_title(f"Filtracja {name}")
    axes[1].imshow(lap_u8, "gray")

    axes[2].axis("off")
    axes[2].set_title("Suma")
    axes[2].imshow(sharp_u8, "gray")

    axes[3].axis("off")
    axes[3].set_title("Modulo")
    axes[3].imshow(diff_u8, "gray")

    plt.show()

compare_laplasjan(moon, M)

### Gradienty (wykorzystanie pierwszej pochodnej obrazu)

1. Wczytaj obraz _kw.png_. Stwórz odpowiednie maski opisane w kolejnych punktach i dokonaj filtracji.
2. Wykorzystując gradient Robertsa przeprowadź detekcję krawędzi - poprzez wykonanie konwolucji obrazu z daną maską:
\begin{equation}
R1 = \begin{bmatrix} 0 & 0 & 0 \\ -1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}   
R2 = \begin{bmatrix} 0 & 0 & 0 \\ 0 & 0 & -1 \\ 0 & 1 & 0 \end{bmatrix}
\end{equation}

Wykorzystaj stworzony wcześniej kod (przy laplasjanie) - dwie metody normalizacji oraz sposób wyświetlania.

3. Analogicznie przeprowadź detekcję krawędzi za pomocą gradientu Prewitta (pionowy i poziomy)
\begin{equation}
P1 = \begin{bmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{bmatrix}   
P2 = \begin{bmatrix} -1 & -1 & -1 \\ 0 & 0 & 0 \\ 1 & 1 & 1 \end{bmatrix}
\end{equation}

4. Podobnie skonstruowany jest gradient Sobela (występuje osiem masek, zaprezentowane są dwie ``prostopadłe''):
\begin{equation}
S1 = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}   
S2 = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}
\end{equation}

Przeprowadź detekcję krawędzi za pomocą gradientu Sobela.

In [ ]:
R1 = np.array([[0, 0, 0], 
               [-1, 0, 0], 
               [0, 1, 0]])

R2 = np.array([[0, 0, 0], 
               [0, 0, -1], 
               [0, 1, 0]])

P1 = np.array([[-1, 0, 1], 
               [-1, 0, 1], 
               [-1, 0, 1]])

P2 = np.array([[-1, -1, -1], 
               [0, 0, 0], 
               [1, 1, 1]])

S1 = np.array([[-1, 0, 1], 
               [-2, 0, 2], 
               [-1, 0, 1]])

S2 = np.array([[-1, -2, -1], 
               [0, 0, 0], 
               [1, 2, 1]])

kernels = {
    "R1" : R1,
    "R2" : R2,
    "P1" : P1,
    "P2" : P2,
    "S1" : S1,
    "S2" : S2,
}

for kernel in kernels.keys():
    compare_filters(images["kw"], kernels[kernel], kernel)

5. Na podstawie dwóch ortogonalnych masek np. Sobela można stworzyć tzw. filtr kombinowany - pierwiastek kwadratowy z sumy kwadratów gradientów:
\begin{equation}
OW = \sqrt{(O * S1)^2 + (O * S2)^2}
\end{equation}
gdzie:  $OW$ - obraz wyjściowy, $O$ - obraz oryginalny (wejściowy), $S1,S2$ - maski Sobela, $*$ - operacja konwolucji.

Zaimplementuj filtr kombinowany.

Uwaga. Proszę zwrócić uwagę na konieczność zmiany formatu danych obrazu wejściowego - na typ znakiem



In [ ]:
S1 = np.array([[-1, 0, 1], 
               [-2, 0, 2], 
               [-1, 0, 1]])

S2 = np.array([[-1, -2, -1], 
               [0, 0, 0], 
               [1, 2, 1]])

def filtr_kombinowany(img, S1, S2):
    moon_u16 = img.astype("int16")

    G_x = cv2.filter2D(moon_u16, cv2.CV_64F, S1)
    G_y = cv2.filter2D(moon_u16, cv2.CV_64F, S2)

    ow = np.sqrt(np.power(G_x, 2) + np.power(G_y, 2))

    return np.clip(ow, 0, 255).astype("uint8")

moon_u8 = filtr_kombinowany(images["kw"], S1, S2)
plt.axis("off")
plt.title("Filt kombinowany")
plt.imshow(moon_u8, "gray")

6. Istnieje alternatywna wersja filtra kombinowanego, która zamiast pierwiastka z sumy kwadratów wykorzystuje sumę modułów (prostsze obliczenia).
Zaimplementuj tę wersję.

In [ ]:
S1 = np.array([[-1, 0, 1], 
               [-2, 0, 2], 
               [-1, 0, 1]])

S2 = np.array([[-1, -2, -1], 
               [0, 0, 0], 
               [1, 2, 1]])

def filtr_kombinowany_easier(img, S1, S2):
    moon_u16 = img.astype("int16")

    G_x = cv2.filter2D(moon_u16, cv2.CV_64F, S1)
    G_y = cv2.filter2D(moon_u16, cv2.CV_64F, S2)

    ow = np.abs(G_x) + np.abs(G_y)

    return np.clip(ow, 0, 255).astype("uint8")


moon_u8_m = filtr_kombinowany_easier(images["kw"], S1, S2)
plt.axis("off")
plt.title("Filt kombinowany modulo")
plt.imshow(moon_u8_m, "gray")

7. Wczytaj plik _jet.png_ (zamiast _kw.png_).
Sprawdź działanie obu wariantów filtracji kombinowanej.

In [ ]:
jet_u8 = filtr_kombinowany(images["jet"], S1, S2)
plt.axis("off")
plt.title("Filt kombinowany")
plt.imshow(jet_u8, "gray")
plt.show()

jet_u8_m = filtr_kombinowany_easier(images["jet"], S1, S2)
plt.axis("off")
plt.title("Filt kombinowany modulo")
plt.imshow(jet_u8_m, "gray")
plt.show()